# Suntime

Discovering time based on the position of the sun.

[Copyright &copy; Anoduck, The Anonymous Duck; 2025](https://anoduck.mit-license.org)

## Previous Work

### Mount Google Drive

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
if not os.path.exists("/content/drive/MyDrive/Colab Notebooks/suntime-opencv/results"):
  os.mkdir("/content/drive/MyDrive/Colab Notebooks/suntime-opencv/results")

# Variables for runtime
batch = False

### Setup Environment

In [ ]:
import os

class myenv:

  def __init__(self) -> None:
    self.setup_os()
    self.setup_env()
    # self.auto_install()
    self.clone_repos()
    # self.create_conda_env()
    self.setup_anyshadow()
    self.init_env()
    self.load_imports()
    self.read_image()

  def setup_os(self):
    import os
    os.chdir("/content")
    global CODE_DIR
    CODE_DIR = "/content/suntime"
    print("Done...")

  def setup_env(self):
    from pathlib import Path
    import os
    import shutil
    if Path(CODE_DIR).exists():
      shutil.rmtree(CODE_DIR)
    !git clone https://github.com/anoduck/suntime $CODE_DIR
    os.chdir(CODE_DIR)
    !git checkout develop
    print(os.listdir(CODE_DIR))
    print("Done...")

  def auto_install(self):
    !pip install -q condacolab
    import condacolab
    condacolab.install()

  # Clone Repository
  def clone_repos(self):
    import shutil
    import os
    global REPO_DIR
    REPO_DIR = "/content/suntime/colab"
    repolist = ["AdapterShadow", "Detect-AnyShadow"]
    print(os.listdir("/content"))
    os.chdir(REPO_DIR)
    for repo in repolist:
      if os.path.exists(os.path.join(REPO_DIR, repo)):
        shutil.rmtree(os.path.join(REPO_DIR, repo))
    !git clone https://github.com/LeipingJie/AdapterShadow $REPO_DIR/AdapterShadow
    !git clone https://github.com/harrytea/Detect-AnyShadow $REPO_DIR/Detect-AnyShadow
    print("Done...")

  def create_conda_env(self):
    !conda init
    !conda create -n suntime python=3.7
    !conda activate suntime

  def setup_anyshadow(self):
    CWD = os.getcwd()
    CDPATH = os.path.abspath(REPO_DIR)
    os.chdir(f"{CDPATH}/Detect-AnyShadow")
    !pip install -r requirements.txt
    os.chdir(CWD)

  def init_env(self):
    # !conda info --envs
    # Updating the environment.
    !pip install -U opencv-python matplotlib pytesseract shadowfinder
    !pip install -U git+https://github.com/pingswept/pysolar
    !pip install -q -U torch torchvision --index-url https://download.pytorch.org/whl/cu121
    # !pip install "optimum-onnx[onnxruntime]"@git+https://github.com/huggingface/optimum-onnx.git
    !pip install -U datasets transformers accelerate timm optimum-onnx optimum[exporters,onnxruntime]
    !pip install -U albumentations pycocotools
    !pip install -U xformers --index-url https://download.pytorch.org/whl/cu121
    !pip install -e ".[torch]"
    print("The environemnt has beed updated...")

  def load_imports(self):
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
    import cv2 as cv
    import random as rng
    import numpy as np
    import random as rng
    from matplotlib import pyplot as plt
    import os
    import google.colab.patches as colab
    from google.colab.patches import cv_imshow as colab_show
    from datasets import load_dataset
    import torch
    import torchvision
    from optimum import onnxruntime as ort
    from optimum.onnxruntime import ORTModelForFeatureExtraction
    from transformers import SamModel, SamProcessor
    from PIL import Image as PIMAGE
    import math
    import glob
    from scipy import ndimage
    import mpl_toolkits.mplot3d.axes3d as p3
    import pandas as pd
    import sys
    import datetime
    import pytz
    import pytesseract
    import shadowfinder
    import cv2 as cv
    print("Done importing modules.")

  def read_image(self):
    img = cv.imread('/content/drive/MyDrive/Colab Notebooks/suntime-opencv/IMAG0692_V1xF8JAe.jpg', 1)

In [ ]:
def env_reload():
  myenv()
  gcenv = myenv()
  gcenv.init_env

env_reload()

In [ ]:
def save_img(cvimg: np.ndarray, label: str) -> bool:
  write_path = os.path.join("/content/drive/MyDrive/Colab Notebooks/suntime-opencv/results", f"{label}.jpg")
  cv.imwrite(write_path, cvimg)
  return True

### Image Utils for Analemma

In [ ]:
class Image:
  @classmethod
  def stackImages(cls, imgArray, scale, lables=None):
      if lables is None:
          lables = []
      sizeW = imgArray[0][0].shape[1]
      sizeH = imgArray[0][0].shape[0]
      rows = len(imgArray)
      cols = len(imgArray[0])
      rowsAvailable = isinstance(imgArray[0], list)
      width = imgArray[0][0].shape[1]
      height = imgArray[0][0].shape[0]
      if rowsAvailable:
          for x in range(0, rows):
              for y in range(0, cols):
                  imgArray[x][y] = cv.resize(imgArray[x][y], (int(sizeW * scale), int(sizeH * scale)))
                  if len(imgArray[x][y].shape) == 2: imgArray[x][y] = cv.cvtColor(imgArray[x][y], cv.COLOR_GRAY2BGR)
          imageBlank = np.zeros((height, width, 3), np.uint8)
          hor = [imageBlank] * rows
          hor_con = [imageBlank] * rows
          for x in range(0, rows):
              hor[x] = np.hstack(imgArray[x])
              hor_con[x] = np.concatenate(imgArray[x])
          try:
              ver = np.vstack(hor)
              ver_con = np.concatenate(hor)
          except:
              pass
      else:
          for x in range(0, rows):
              imgArray[x] = cv.resize(imgArray[x], (int(sizeW * scale), int(sizeH * scale)))
              if len(imgArray[x].shape) == 2: imgArray[x] = cv.cvtColor(imgArray[x], cv.COLOR_GRAY2BGR)
          hor = np.hstack(imgArray)
          hor_con = np.concatenate(imgArray)
          ver = hor
      if len(lables) != 0:
          eachImgWidth = int(ver.shape[1] / cols)
          eachImgHeight = int(ver.shape[0] / rows)
          for d in range(0, rows):
              for c in range(0, cols):
                  cv.rectangle(ver, (c * eachImgWidth, eachImgHeight * d),
                                (c * eachImgWidth + len(lables[d][c]) * 13 + 27, 30 + eachImgHeight * d),
                                (255, 255, 255), cv.FILLED)
                  cv.putText(ver, lables[d][c], (eachImgWidth * c + 10, eachImgHeight * d + 20),
                              cv.FONT_HERSHEY_COMPLEX, 0.7, (255, 0, 255), 2)
      return ver

  @classmethod
  def warp(cls, dst: np.ndarray, transformation_matrix: list) -> np.ndarray:
      w, h = dst.shape[:2]
      return cv.warpPerspective(dst, transformation_matrix, (w, h))

  @classmethod
  def get_formated_canny(cls, image: np.ndarray) -> np.ndarray:
      """
      Formats an image to gray, blur, and lastly to canny

      :param image: Source image
      :return: Canny image
      """
      img = image.copy()
      gray = Image.cvt_to_gray(img)
      blur = cv.GaussianBlur(gray, (5, 5), 1)
      canny = cv.Canny(blur, 10, 50)
      return canny

  @classmethod
  def size_reduction(cls, canvas: np.ndarray, size_reduction: float) -> np.ndarray:
      """
      | Reduces image size by cutting of a percentage of pixels starting from the image outlines

      :param canvas: Source image
      :param size_reduction: Percentage of pixels that gets cut of
      """
      canvas = Image.cvt_to_gray(canvas)
      h, w = canvas.shape[:2]
      reduce_pixels_h = int(((h / 100) * size_reduction) / 2)
      reduce_pixels_w = int(((w / 100) * size_reduction) / 2)

      x = reduce_pixels_w
      w = w - reduce_pixels_w
      y = reduce_pixels_h
      h = h - reduce_pixels_h
      return canvas[y:h, x:w]

  @classmethod
  def cvt_to_gray(cls, image: np.ndarray) -> np.ndarray:
      image = image.copy()
      if len(image.shape) < 3:
          return image

      channels = image.shape[2]
      match channels:
          case 3:
              try:
                  image = cv.cvtColor(image, cv.COLOR_BGR2GRAY)
              except:
                  try:
                      image = cv.cvtColor(image, cv.COLOR_RGB2GRAY)
                  except:
                      try:
                          image = cv.cvtColor(image, cv.COLOR_HSV2BGR)
                          image = cv.cvtColor(image, cv.COLOR_RGB2GRAY)
                      except:
                          print("Image format not supported")
                          assert ValueError
      return image

  @classmethod
  def show(cls, img: np.ndarray, winname="test", destroy=False) -> None:
      cv.imshow(winname, img)
      cv.waitKey(99999999)
      if destroy:
          cv.destroyWindow(winname)

### Analemma

In [ ]:
def analemma(image) -> tuple:
  radius = int(51)
  last_center = (0.0, 0.0)
  # reduce image size by 20px on all sides and auto converts it to gray
  (h, w) = image.shape[:2]
  h2 = h - 20
  w2 = w - 20
  image_copy = cv.resize(image, (w2, h2))
  image = Image.size_reduction(image, 20)
  print(f"Image copy shape: {image_copy.shape} and image shape: {image.shape}")
  # insure radius is odd
  print(f"Radius: {radius}")
  if int(radius) % 2:
      pass
  else:
      radius += 1
  # blur the image
  grey = Image.cvt_to_gray(image)
  try:
      blur = cv.GaussianBlur(grey, (int(radius), int(radius)), cv.BORDER_DEFAULT)
  except Exception:
      blur = cv.medianBlur(grey, int(radius))

  # calculate minMax method
  minMaxMethod = image.copy()
  # grey = cv.cvtColor(img, cv.COLOR_BGR2GRAY)
  # method = cv.TM_CCOEFF_NORMED
  # res = cv.matchTemplate(img,template,method)
  (minVal, maxVal, minLoc, maxLoc) = cv.minMaxLoc(blur)
  minMaxCenter = maxLoc
  # apply the minMax method
  cv.circle(minMaxMethod, maxLoc, int(radius), (0, 0, 0), 2)
  # prepare the image for the robust method
  thresh = cv.threshold(blur, 210, 225, cv.THRESH_BINARY)[1]
  erode = cv.erode(thresh, None, iterations=7)
  dilate = cv.dilate(erode, None, iterations=4)
  canny = Image.get_formated_canny(dilate)
  # calculate robust method
  points = np.argwhere(canny > 0)
  robustCenter, radius = cv.minEnclosingCircle(points)
  # apply robust method
  robustMethod = image.copy()
  x = int(robustCenter[1])
  y = int(robustCenter[0])
  rad = int(radius)
  cv.circle(robustMethod, (x, y), rad, (300, 100, 100), 2)
  # debug print
  print("lastCenter: " + str(last_center))
  print("dist: " + str(math.dist(robustCenter, last_center)))
  print("robustCenter: " + str(robustCenter))
  print("maxLoc: " + str(robustCenter))
  print("----------------------------------")
  # determine which method to use
  center = minMaxCenter
  if robustCenter == (0.0, 0.0):
      if last_center != (0.0, 0.0):
          if not (math.dist(minMaxCenter, last_center) < 50):
              center = robustCenter
      else:
          center = (0.0, 0.0)
  # put text and highlight the center
  Cx, Cy = center
  image1 = cv.circle(image_copy, (Cx, Cy), 5, (0, 255, 12), -1)
  image2 = cv.putText(
      image1,
      "centroid",
      (Cx - 25, Cy - 25),
      cv.FONT_HERSHEY_SIMPLEX,
      2,
      (0, 255, 12),
      2,
  )
  print(f"Center: {center}, Radius: {int(radius)}")
  colab_show(image2)
  return image2, center

# img = cropped_img
sunid_img, center = analemma(img)

### Use PyTorch, ONNX, and SAM to detect shadows

In [ ]:
class TorchShadows:
  def __init__(self, cvimg):
    self.cvimg = cv.cvtColor(cvimg, cv.COLOR_BGR2LAB)
    self.model_id = "facebook/sam-vit-huge"
    self.model = SamModel.from_pretrained(self.model_id)
    self.processor = SamProcessor.from_pretrained(self.model_id)
    self.ort_model = ORTModelForFeatureExtraction.from_pretrained(self.model_id, export=True)
    self.ort_session = ort.InferenceSession(self.ort_model.model_save_path / "model.onnx")
    # To use TPU if available, otherwise fallback to CPU
    # try:
    #   self.device = xm.xla_device()
    # except RuntimeError: # If XLA is not available or TPU not found
    self.device = torch.device("gpu")
    self.model.to(self.device)
    self.dataset = load_dataset("emasquil/shadow-eo")

  def translate_image(self, matlike) -> PIMAGE:
    if matlike is None:
      print("Error: Could not load image for translation.")
    else:
      # 2. Convert from BGR (OpenCV) to RGB (Pillow)
      color_converted_image = cv.cvtColor(matlike, cv.COLOR_BGR2RGB)
      # 3. Create Pillow Image object from the NumPy array
      pil_image = PIMAGE.fromarray(color_converted_image)
      # Now 'pil_image' is a Pillow Image object, you can use Pillow's functions
      return pil_image.convert("RGB")

  def discover_points(self, raw_image):
    gray = cv.cvtColor(np.array(raw_image), cv.COLOR_RGB2GRAY)
    _, thresh = cv.threshold(gray, 50, 255, cv.THRESH_BINARY_INV)  # Threshold for dark areas
    contours, _ = cv.findContours(thresh, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)
    if contours:
      # Get center of largest contour as prompt point
      largest_contour = max(contours, key=cv.contourArea)
      M = cv.moments(largest_contour)
      cx = int(M['m10'] / M['m00'])
      cy = int(M['m01'] / M['m00'])
      self.input_points = [[[cx, cy]]]

  def segment_image(self):
    raw_image = self.translate_image(self.cvimg)
    self.discover_points(raw_image)
    # Process inputs
    self.inputs = self.processor(raw_image, input_points=self.input_points, return_tensors="pt")  # Use input_boxes=input_boxes for boxes
    # Run inference
    with torch.no_grad():
      outputs = self.model(**self.inputs)
    # Post-process masks
    masks = self.processor.image_processor.post_process_masks(
      outputs.pred_masks.cpu(),
      self.inputs["original_sizes"].cpu(),
      self.inputs["reshaped_input_sizes"].cpu()
    )[0]  # Get the first set of masks
    # Visualize the best mask (highest IoU score)
    self.best_mask = masks[0][outputs.iou_scores.argmax().item()].numpy()  # Select mask with highest score
    plt.imshow(np.array(raw_image))
    plt.imshow(self.best_mask, alpha=0.5, cmap='gray')  # Overlay mask on image (white for shadow)
    plt.title("Segmented Shadow Area")
    plt.show()
    # Save the mask as an image if needed
    mask_image = (self.best_mask * 255).astype(np.uint8)
    save_img(mask_image, "shadow_mask")

  def image_interference(self):
    self.segment_image()
    ort_inputs = {k: v for k, v in self.inputs.items() if k in self.ort_session.get_inputs()}  # Filter valid inputs
    outputs = self.ort_session.run(None, ort_inputs)
    # Post-process (adapt from transformers post-processing)
    # Note: ONNX outputs may need manual handling; use NumPy for masks
    pred_masks = outputs[0]  # Assuming first output is pred_masks
    # Apply sigmoid and threshold for binary mask (adjust as needed)
    masks = (pred_masks > 0.5).astype(np.uint8)  # Simplified; use full post-processing logic from transformers if needed

    # Visualize or save as before
    mask_image = Image.fromarray(masks[0][0] * 255)  # Example for first mask
    self.onnx_cvimg = cv.cvtColor(np.array(mask_image), cv.COLOR_GRAY2BGR)
    save_img(self.onnx_cvimg, "onnx_shadow_mask")

In [ ]:
ts = TorchShadows(img)
ts.image_interference()

### Find Shadows

In [ ]:
class HSVRun:
  def __init__(self):
      pass

  def midpoint(self, ptA, ptB):
      """Helper to compute midpoint between two points."""
      return ((ptA[0] + ptB[0]) * 0.5, (ptA[1] + ptB[1]) * 0.5)

  def detect_shape(self, contour):
      """Detect simple shapes based on approximated contour."""
      shape = "unidentified"
      peri = cv.arcLength(contour, True)
      approx = cv.approxPolyDP(contour, 0.04 * peri, True)
      if len(approx) == 3:
          shape = "triangle"
      elif len(approx) == 4:
          (x, y, w, h) = cv.boundingRect(approx)
          ar = w / float(h)
          shape = "square" if 0.95 <= ar <= 1.05 else "rectangle"
      elif len(approx) == 5:
          shape = "pentagon"
      else:
          shape = "circle"  # or ellipse
      return shape

  def discover_shadows(self, cv_image):
    # Optional: Normalize to reduce uneven lighting/shadows (division normalization)
    gray = cv.cvtColor(cv_image, cv.COLOR_BGR2GRAY)
    blur = cv.GaussianBlur(gray, (95, 95), 0)
    normalized = cv.divide(gray, blur, scale=255)

    # Convert to HSV for thresholding
    hsv = cv.cvtColor(cv_image, cv.COLOR_BGR2HSV)
    h, s, _ = cv.split(hsv)
    # Merge normalized grayscale as the Value channel
    hsv_normalized = cv.merge([h, s, normalized])
    # Convert back to BGR for visualization (optional)
    image_normalized = cv.cvtColor(hsv_normalized, cv.COLOR_HSV2BGR)

    # Threshold for shadows: Low Value (adjust thresholds based on image; e.g., V < 100 for dark areas)
    shadow_lower = np.array([0, 0, 0])
    shadow_upper = np.array([180, 255, 100])  # Low V for shadows
    shadow_mask = cv.inRange(hsv_normalized, shadow_lower, shadow_upper)

    # Threshold for objects: Higher Value (brighter regions)
    object_lower = np.array([0, 0, 150])  # High V for non-shadows
    object_upper = np.array([180, 255, 255])
    object_mask = cv.inRange(hsv_normalized, object_lower, object_upper)

    # Ensure shadow areas are closed: Apply morphological closing to fill gaps/holes
    kernel = np.ones((5, 5), np.uint8)  # Adjust kernel size as needed
    shadow_mask = cv.morphologyEx(shadow_mask, cv.MORPH_CLOSE, kernel, iterations=2)

    # Find contours for shadows
    shadow_contours, _ = cv.findContours(shadow_mask, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)

    # Find contours for objects
    object_contours, _ = cv.findContours(object_mask, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)

    # Process shadows (outline, label, identify object, measure)
    print(cv_image.shape)
    output = image_normalized.copy()  # Use normalized image for output
    CY = cv_image.shape[0]
    CX = cv_image.shape[1]
    lg_side = max(CX, CY)
    print(f"CX: {CX}, CY: {CY}, lg_side: {lg_side}")
    sm_side = min(CX, CY)
    print(f"CX: {CX}, CY: {CY}, sm_side: {sm_side}")
    # min_area = 800  # Ignore small contours; adjust as needed
    max_area = sm_side * lg_side / 16
    min_area = sm_side * lg_side / sm_side
    object_centers = []  # Store object centers and shapes

    # Collect object info
    for oc in object_contours:
        if cv.contourArea(oc) > min_area:
            M = cv.moments(oc)
            if M["m00"] != 0:
                cx = int(M["m10"] / M["m00"])
                cy = int(M["m01"] / M["m00"])
                shape = self.detect_shape(oc)
                object_centers.append(((cx, cy), shape))

    # Process each shadow contour
    print(f"Shadow contours: {len(shadow_contours)}")
    for sc in shadow_contours:
        area = cv.contourArea(sc)
        if min_area < area < max_area:  # Now checks both min and max area
            # Outline the shadow shape
            cv.drawContours(output, [sc], -1, (255, 0, 255), 2)  # Cyan outline

            # Label as "Shadow"
            M = cv.moments(sc)
            if M["m00"] != 0:
                cx = int(M["m10"] / M["m00"])
                cy = int(M["m01"] / M["m00"])
                cv.putText(output, "Shadow", (cx - 20, cy - 10), cv.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2)

                # Identify corresponding object: Find nearest object by Euclidean distance
                if object_centers:
                    distances = [np.linalg.norm(np.array((cx, cy)) - np.array(oc[0])) for oc in object_centers]
                    nearest_idx = np.argmin(distances)
                    object_shape = object_centers[nearest_idx][1]
                    cv.putText(output, f"From: {object_shape}", (cx - 20, cy + 10), cv.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)

                # Measure shadow length: Fit ellipse and get major axis (or use bounding box for simplicity)
                if len(sc) >= 5:  # Ellipse needs at least 5 points
                    ellipse = cv.fitEllipse(sc)
                    major_axis = max(ellipse[1])  # Longer dimension
                else:
                    x, y, w, h = cv.boundingRect(sc)
                    major_axis = max(w, h)
                cv.putText(output, f"Len: {major_axis:.1f}px", (cx - 20, cy + 30), cv.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
    return output

hsv = HSVRun()
shadowed = hsv.discover_shadows(sunid_img)
colab_show(shadowed)
cv.imwrite("/content/drive/MyDrive/Colab Notebooks/suntime-opencv/hsvrun.jpg", shadowed)

### Alternative Thresholding Approach

In [ ]:
class AdaptiveShadow:

  def __init__(self, cvimg):
    self.cvimg = cvimg

  def colorspace(self):
    grayed = cv.cvtColor(self.cvimg, cv.COLOR_BGR2GRAY)
    dilated = cv.dilate(grayed, np.ones((7,7), np.uint8))
    bg_img = cv.medianBlur(dilated, 21)
    diff_img = 255 - cv.absdiff(grayed, bg_img)
    normalized = cv.normalize(diff_img, None, alpha=0, beta=255, norm_type=cv.NORM_MINMAX, dtype=cv.CV_8UC1)
    result = cv.cvtColor(normalized, cv.COLOR_GRAY2BGR)
    cv.imwrite("/content/drive/MyDrive/Colab Notebooks/suntime-opencv/adaptived.jpg", result)
    plt.imshow(result)
    plt.show()
    return result